[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E5%88%9D%E8%A6%8Bpandas/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E6%96%B0%E5%93%81%E4%B8%8A%E5%B8%82%E9%80%B1%E7%9A%84%E7%AC%AC%E4%B8%80%E4%BB%BD%E5%A0%B1%E8%A1%A8.ipynb)

# 🐼 茶語時光：新品上市週的第一份 pandas 報表

> 🧋 **情境**：2024 年 4 月 1 日，新品「芋頭鮮奶」上市！行銷部想知道：新品賣得怎樣？有沒有搶走其他飲料的生意？每杯賺多少？你第一次用 pandas，把菜單、門市資料與兩週的銷量做成表格來回答。

| 素材檔案 | 內容 |
| --- | --- |
| `菜單.csv` | 10 個品項的價格與成本 |
| `門市.csv` | 5 家門市的基本資料 |
| `兩週品項銷量.json` | 芋頭鮮奶上市前一週與上市當週，全品牌各品項杯數 |
| `信義店日報_新品週.json` | 信義店新品週每天的杯數、營收、來客數 |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '菜單.csv',
    '門市.csv',
    '兩週品項銷量.json',
    '信義店日報_新品週.json',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E5%88%9D%E8%A6%8Bpandas/%E5%88%9D%E8%A6%8Bpandas_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['菜單.csv', '門市.csv', '兩週品項銷量.json', '信義店日報_新品週.json']


## 任務 1：Series——有名牌的一欄數字

**老闆的問題**：「每杯中杯賣多少錢？超過 60 元的有哪些？如果全面漲 5%，會變多少？」

**白話說明**：Series 像 Excel 的一欄，左邊是名牌（品項），右邊是數值（價格）。

In [2]:
import pandas as pd

menu = pd.read_csv('菜單.csv', dtype={'品項代碼': str})     # 品項代碼當文字讀，才不會把 001 變成 1
price = pd.Series(menu['中杯價'].values, index=menu['品項'], name='中杯價')   # 名牌＝品項，值＝價格
print(price)
print('\n珍珠奶茶的價格：', price['珍珠奶茶'])               # 用名牌查詢
print('\n超過 60 元的品項：\n', price[price > 60])           # 篩選
print('\n全面漲 5% 後：\n', (price * 1.05).round())         # 整欄一起運算

品項
珍珠奶茶      55
四季春青茶     35
鮮奶綠       55
冬瓜檸檬      50
多多綠       50
黑糖珍珠鮮奶    75
芒果冰沙      70
熱可可       60
檸檬紅茶      40
芋頭鮮奶      70
Name: 中杯價, dtype: int64

珍珠奶茶的價格： 55

超過 60 元的品項：
 品項
黑糖珍珠鮮奶    75
芒果冰沙      70
芋頭鮮奶      70
Name: 中杯價, dtype: int64

全面漲 5% 後：
 品項
珍珠奶茶      58.0
四季春青茶     37.0
鮮奶綠       58.0
冬瓜檸檬      52.0
多多綠       52.0
黑糖珍珠鮮奶    79.0
芒果冰沙      74.0
熱可可       63.0
檸檬紅茶      42.0
芋頭鮮奶      74.0
Name: 中杯價, dtype: float64


## 任務 2：DataFrame——加上毛利欄位，找出「最會賺」的飲料

**白話說明**：DataFrame 就是整張 Excel 表。新增欄位就像在 Excel 插入一欄、填入公式。

In [3]:
menu['毛利'] = menu['中杯價'] - menu['成本']                 # 新增欄位：每杯賺多少
menu['毛利率'] = (menu['毛利'] / menu['中杯價'] * 100).round(1)
print(menu)

print('\n形狀：', menu.shape)                                # (列數, 欄數)
print('\n毛利率 > 70% 的品項：')
print(menu[menu['毛利率'] > 70][['品項', '中杯價', '毛利', '毛利率']])

  品項代碼      品項  類別  中杯價  成本  毛利   毛利率
0  001    珍珠奶茶  奶茶   55  17  38  69.1
1  002   四季春青茶  純茶   35   7  28  80.0
2  003     鮮奶綠  鮮奶   55  20  35  63.6
3  004    冬瓜檸檬  果茶   50  14  36  72.0
4  005     多多綠  果茶   50  15  35  70.0
5  006  黑糖珍珠鮮奶  鮮奶   75  27  48  64.0
6  007    芒果冰沙  冰沙   70  29  41  58.6
7  008     熱可可  熱飲   60  21  39  65.0
8  009    檸檬紅茶  果茶   40  11  29  72.5
9  010    芋頭鮮奶  鮮奶   70  26  44  62.9

形狀： (10, 7)

毛利率 > 70% 的品項：
      品項  中杯價  毛利   毛利率
1  四季春青茶   35  28  80.0
3   冬瓜檸檬   50  36  72.0
8   檸檬紅茶   40  29  72.5


> 💡 **看懂結果**：純茶、果茶的毛利率最高（成本低），鮮奶系列單價高、每杯賺得多但毛利率較低。**促銷要推哪一杯，取決於你要衝「營收」還是衝「利潤率」。**
>
> 🤖 **換你問 AI**：「這是我的菜單 DataFrame（欄位：品項、中杯價、成本），請新增毛利與毛利率欄，並列出毛利率超過 70% 的品項。」

## 任務 3：從 JSON 建立 DataFrame——信義店新品週日報

**白話說明**：「清單裡放字典」（一天一個字典）的 JSON，用 `pd.DataFrame()` 一行就變成表格。

In [4]:
import json

with open('信義店日報_新品週.json', encoding='utf-8') as f:
    records = json.load(f)
print('原始資料第一筆：', records[0])

report = pd.DataFrame(records)                               # 一個字典 → 一列
report = report.set_index('日期')                             # 用日期當列名牌
report['客單價'] = (report['營收'] / report['來客數']).round(1) # 平均每位客人花多少錢
print(report)
print('\n新品週總營收：', report['營收'].sum(), '元')
print('雨天那天：\n', report.loc[report['天氣'] == '雨'])

原始資料第一筆： {'日期': '2024-04-01', '天氣': '晴', '杯數': 499, '營收': 27576, '來客數': 336}
            天氣   杯數     營收  來客數   客單價
日期                                   
2024-04-01   晴  499  27576  336  82.1
2024-04-02  多雲  443  25099  286  87.8
2024-04-03   晴  387  21034  241  87.3
2024-04-04   雨  310  17604  195  90.3
2024-04-05   晴  419  23389  272  86.0
2024-04-06  多雲  309  17981  190  94.6
2024-04-07   晴  248  14299  159  89.9

新品週總營收： 146982 元
雨天那天：
            天氣   杯數     營收  來客數   客單價
日期                                  
2024-04-04  雨  310  17604  195  90.3


## 任務 4：新品有沒有搶走其他飲料的生意？——索引對齊與 NaN

**老闆的問題**：「芋頭鮮奶上市後，其他飲料有沒有變少？」

**白話說明**：兩個 Series 相減時，pandas 會「按名牌配對」。芋頭鮮奶上週還沒上市、沒有名牌，配不到就會出現 **NaN**（空值）——這正是新品！

In [5]:
with open('兩週品項銷量.json', encoding='utf-8') as f:
    sales = json.load(f)

last_week = pd.Series(sales['上週(3/25-3/31)'])               # 上週：沒有芋頭鮮奶
this_week = pd.Series(sales['本週(4/1-4/7)'])                 # 本週：多了芋頭鮮奶

compare = pd.DataFrame({'上週': last_week, '本週': this_week})  # 依品項名稱自動對齊
compare['變化'] = compare['本週'] - compare['上週']             # 上週沒有的 → NaN
compare['變化率%'] = (compare['變化'] / compare['上週'] * 100).round(1)
print(compare.sort_values('變化'))
print('\n上週沒有資料（新品）：', list(compare[compare['上週'].isnull()].index))

            上週      本週     變化  變化率%
黑糖珍珠鮮奶  1840.0  1511.0 -329.0 -17.9
鮮奶綠     1536.0  1282.0 -254.0 -16.5
多多綠     1232.0  1218.0  -14.0  -1.1
檸檬紅茶    1343.0  1336.0   -7.0  -0.5
冬瓜檸檬    1379.0  1386.0    7.0   0.5
四季春青茶   1843.0  1851.0    8.0   0.4
珍珠奶茶    2932.0  2944.0   12.0   0.4
芒果冰沙     380.0   491.0  111.0  29.2
熱可可       66.0     NaN    NaN   NaN
芋頭鮮奶       NaN  1624.0    NaN   NaN

上週沒有資料（新品）： ['芋頭鮮奶']


> 💡 **看懂結果**：
> - 芋頭鮮奶的「變化」是 NaN，因為上週根本還沒賣；熱可可則相反，天氣變熱、本週沒人點。**NaN 不是錯誤，而是在告訴你「這裡沒有資料可以比」**。
> - 同樣是鮮奶系列的「黑糖珍珠鮮奶」、「鮮奶綠」是衰退最多的兩杯——客人很可能改點新品了，這叫「**自我侵蝕**」。新品賣得好，不代表品牌整體多賺了這麼多。
>
> 🤖 **換你問 AI**：「我有上週與本週各品項銷量兩個 Series，請合併成一張比較表，計算變化量與變化率，並說明 NaN 代表什麼。」

## 任務 5：門市資料表——每個座位要負擔多少租金？

In [6]:
stores = pd.read_csv('門市.csv')
stores['每座位月租金'] = (stores['月租金'] / stores['座位數']).round(0)   # 新增欄位
print(stores[['門市', '縣市', '座位數', '月租金', '每座位月租金']].sort_values('每座位月租金', ascending=False))

      門市   縣市  座位數     月租金   每座位月租金
1  台北西門店  台北市    8  150000  18750.0
0  台北信義店  台北市   12  180000  15000.0
2  台中逢甲店  台中市    6   90000  15000.0
4  高雄巨蛋店  高雄市   10   85000   8500.0
3  台南成大店  台南市   20   60000   3000.0


> 💡 **看懂結果**：台北的門市每個座位的租金是台南的好幾倍——這也是為什麼台北店更重視「外帶」與「翻桌率」。

---

## ✍️ 練習題

1. 用 `menu` 找出「類別」是「鮮奶」的所有品項，並算出它們的平均毛利。
2. 在 `report` 新增一欄「每杯平均價」＝營收 ÷ 杯數，哪一天最高？
3. 在 `compare` 中，用 `fillna(0)` 把上週的 NaN 補成 0，再重新計算芋頭鮮奶的「變化」。

## 🔑 參考解答（先自己試試看！）

In [7]:
# 練習 1
milk = menu[menu['類別'] == '鮮奶']
print(milk[['品項', '毛利']], '\n平均毛利：', milk['毛利'].mean())

# 練習 2
report['每杯平均價'] = (report['營收'] / report['杯數']).round(1)
print(report['每杯平均價'], '\n最高：', report['每杯平均價'].idxmax())

# 練習 3
compare['上週'] = compare['上週'].fillna(0)
compare['變化'] = compare['本週'] - compare['上週']
print(compare.loc['芋頭鮮奶'])

       品項  毛利
2     鮮奶綠  35
5  黑糖珍珠鮮奶  48
9    芋頭鮮奶  44 
平均毛利： 42.333333333333336
日期
2024-04-01    55.3
2024-04-02    56.7
2024-04-03    54.4
2024-04-04    56.8
2024-04-05    55.8
2024-04-06    58.2
2024-04-07    57.7
Name: 每杯平均價, dtype: float64 
最高： 2024-04-06
上週         0.0
本週      1624.0
變化      1624.0
變化率%       NaN
Name: 芋頭鮮奶, dtype: float64
